# Using a Local Database with SQLite

SQLite is a lightweight, file-based relational database. It's built into Python via the `sqlite3` standard library module — no installation required.

## Popular Relational Databases

- **SQLite** — file-based, great for local development and small apps
- **PostgreSQL** — powerful open-source server database
- **MySQL / MariaDB** — widely used in web applications
- **Microsoft SQL Server** — enterprise-grade database

All of them speak SQL (Structured Query Language), so core skills transfer across them.

## Using sqlite3 in 5 Steps

1. Import `sqlite3`
2. Connect to a database file (creates it if it doesn't exist)
3. Create a cursor
4. Execute SQL and fetch results
5. Close the cursor and connection

In [1]:
import sqlite3

# Create the events table so subsequent cells can query it
with sqlite3.connect("countdown.db") as cn:
    cursor = cn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS events (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            event_name TEXT NOT NULL,
            event_date TEXT,
            priority INTEGER DEFAULT 5,
            is_private INTEGER DEFAULT 0
        )
    """)
    cn.commit()
    cursor.close()

print("Table ready")

Table ready


In [2]:
import sqlite3

cn = sqlite3.connect("countdown.db")
cursor = cn.cursor()

cursor.execute("SELECT * FROM events")
results = cursor.fetchall()
print(results)

cursor.close()
cn.close()

[]


## Using a `with` Block (Recommended)

A `with` block automatically handles closing the connection.

In [3]:
import sqlite3

with sqlite3.connect("countdown.db") as cn:
    cursor = cn.cursor()
    cursor.execute("SELECT * FROM events")
    results = cursor.fetchall()
    print(results)
    cursor.close()

[]


## SQL Operations that Modify Data

For INSERT, UPDATE, and DELETE statements you must call `commit()` after `execute()` to persist the changes.

In [4]:
import sqlite3

with sqlite3.connect("countdown.db") as cn:
    cursor = cn.cursor()
    cursor.execute(
        "INSERT INTO events (event_name, event_date, priority, is_private) VALUES (?, ?, ?, ?)",
        ("Birthday Party", "2024-06-15", 8, 0)
    )
    cn.commit()
    print("Row inserted successfully")
    cursor.close()

Row inserted successfully


## Row Factories

By default, `sqlite3` returns results as a list of tuples. A row factory transforms those tuples into a more convenient structure.

- `sqlite3.Row` — built-in generic class that gives dictionary-like access
- Custom factory function — maps each row tuple to your own dataclass

In [5]:
import sqlite3

# Using the built-in sqlite3.Row factory
with sqlite3.connect("countdown.db") as cn:
    cn.row_factory = sqlite3.Row
    cursor = cn.cursor()
    cursor.execute("SELECT * FROM events")
    results = cursor.fetchall()

    if results:
        # Row objects support dictionary-style key access
        print(results[0]["event_name"])

    cursor.close()

Birthday Party


## From Rows to Python Objects (Custom Row Factory)

Define a factory function that accepts `(cursor, row)` and returns a dataclass instance.

In [6]:
import sqlite3
from dataclasses import dataclass

@dataclass
class Event:
    id: int
    event_name: str
    event_date: str
    priority: int
    is_private: bool

def event_factory(cursor, row):
    return Event(
        id=int(row[0]),
        event_name=row[1],
        event_date=row[2],
        priority=int(row[3]),
        is_private=bool(row[4]),
    )

with sqlite3.connect("countdown.db") as cn:
    cn.row_factory = event_factory
    cursor = cn.cursor()
    cursor.execute("SELECT * FROM events")
    events = cursor.fetchall()  # already a list of Event instances
    for event in events:
        print(event)
    cursor.close()

Event(id=1, event_name='Birthday Party', event_date='2024-06-15', priority=8, is_private=False)


## Let's Build a Countdown Application

Putting it all together — a small app that stores and retrieves countdown events.

In [7]:
import sqlite3
from dataclasses import dataclass
from datetime import date

@dataclass
class Event:
    id: int
    event_name: str
    event_date: str
    priority: int
    is_private: bool

def event_factory(cursor, row):
    return Event(
        id=int(row[0]),
        event_name=row[1],
        event_date=row[2],
        priority=int(row[3]),
        is_private=bool(row[4]),
    )

def setup_db(db_path="countdown.db"):
    with sqlite3.connect(db_path) as cn:
        cursor = cn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS events (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                event_name TEXT NOT NULL,
                event_date TEXT,
                priority INTEGER DEFAULT 5,
                is_private INTEGER DEFAULT 0
            )
        """)
        cn.commit()
        cursor.close()

def add_event(name, event_date, priority=5, is_private=False, db_path="countdown.db"):
    with sqlite3.connect(db_path) as cn:
        cursor = cn.cursor()
        cursor.execute(
            "INSERT INTO events (event_name, event_date, priority, is_private) VALUES (?, ?, ?, ?)",
            (name, str(event_date), priority, int(is_private))
        )
        cn.commit()
        cursor.close()

def list_events(db_path="countdown.db"):
    with sqlite3.connect(db_path) as cn:
        cn.row_factory = event_factory
        cursor = cn.cursor()
        cursor.execute("SELECT * FROM events ORDER BY event_date")
        results = cursor.fetchall()
        cursor.close()
    return results

# Demo
setup_db()
add_event("New Year", "2026-01-01", priority=10)
add_event("Team Outing", "2025-11-15", priority=7)

for event in list_events():
    print(f"{event.event_date} — {event.event_name} (priority {event.priority})")

2024-06-15 — Birthday Party (priority 8)
2025-11-15 — Team Outing (priority 7)
2026-01-01 — New Year (priority 10)
